In [239]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np


In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [241]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.xlsx') in f and not f.startswith('~$'):
                    file_list.append(f)
#file_list

In [242]:
len(file_list)

90

In [243]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

#file_link

In [244]:
cols=['Brand']
df_a = pd.DataFrame(columns=cols)

In [245]:
RemoveColumn=["(B03) Hazardous Material Code",
"(B05) Base Item ID",
"(B29) VMRS Brand ID",
"(B32) Item Qty Size (Each)",
"(B34) Container Type",
"(B35) Vehicle Quantity Qualifier",
"(B40) Qty Per Applications (Each)",
"(B40) Quantity Per Application (Package)",
"(B45) Effective Date",
"(B50) Available Date",
"(B55) Minimum Order Qty (Each)",
"(B60) Group",
"(B61) Sub Group",
"(B62) AAIA Product Category Code",
"(B63) UNSPSC",
"(B65) VMRS Code",
"(C10) ABR - Product Description - Abbreviated - 12",
"(C10) APS- Application Summary - 240",
"(C10) ASC - Associated Comments - 2000",
"(C10) DES - Product Description - Long - 80",
"(C10) EXT - Product Description - Extended 240",
"(C10) FAB - Features and Benefits - 240",
"(C10) INV - Product Description - Invoice - 40",
"(C10) KSW - Key Search Word - 80",
"(C10) LAB - Label Description - 80",
"(C10) MKT - Marketing Description - 2000",
"(C10) SHO - Product Description - Short - 20",
"(C10) SLD - Slang Description - 80",
"(E05) CTO - Country of Origin (Primary)",
"(E05) CTP - Country of Origin 2",
"(E05) CTQ - Country of Origin 3",
"(E05) CTR - Country of Origin 4",
"(E05) MSR - SDR Required Flag",
"(E05) NAF - NAFTA Preference Criterion Code",
"(E05) NPC - National Popularity Code",
"(E05) STA - Stock Status",
"(E10) CCL - Core Class",
"(E10) CGR - Core Group",
"(E10) CPN - Core Part Number",
"(E10) CXP - Core Return Days to Expiry",
"(E10) ECN - ECCN",
"(E10) EMS - Emission Code",
"(E10) HAC - Canadian Harmonizing Tariff Code",
"(E10) HSB - Harmonized Tariff Code (Schedule B)",
"(E10) HTS - Harmonized Tariff Code (HTS)",
"(E10) HZ1 - Item (SKU) Level Special Handling Code",
"(E10) LTM - Estimated Lead Time",
"(E10) MSD - SDS Sheet Number",
"(E10) OEM - Original Equipment Manufacturer",
"(E10) OEP - OEMΓÇÖs Part Number",
"(E10) OSN - Original Supplier",
"(E10) OSP - Original Supplier Part Number",
"(E10) PLC - Maximum Cases per Pallet Layer",
"(E10) PLM - Pallet Layer Maximum",
"(E10) RCC - Regulating Country",
"(E10) RCS - Regulating County, State",
"(E10) RCT - Regulating City, State",
"(E10) RDE - Regulating Description",
"(E10) RET - Return Code",
"(E10) RPC - Regulating Postal Code",
"(E10) RPQ - Return Pack Size",
"(E10) RST - Regulating State",
"(E10) TAX - Taxable",
"(E10) TMC - Trading Partner Movement Code",
"(E10) VMI - Alert Code",
"(E10) WD1 - Warranty Distance",
"(E10) WHS - Supplier Warehouse ID",
"(E10) WS1 Warranty Special",
"(GCT) - Green Certification",
"(GPV) - Additional Green Attributes",
"(PRC) - Product Condition",
"Article Status Code",
"Product Group Overrides",
"Product Line",
"Publishing Countries",
"Region",
]

In [246]:
IDColumns=['Article Number', 'Brand', 'Product Group', 'Article Status Description']

In [247]:
len(file_link)

90

In [248]:
for i in range(len(file_link)):
    #print(f'Processing file: {file_list[i]}')
    df=pd.read_excel(file_link[i],sheet_name='article-export')
    
    try:
        df_dropped = df.drop(RemoveColumn, axis=1)
    except:
        print(f'Columns not found in {file_list[i]}')
    try:
        df=df_dropped.melt(id_vars=IDColumns, var_name='Attribute', value_name='Value')
    except:
        print(f'ID Columns not found in {file_list[i]}')
    df['Article Number']= df['Article Number'].astype(str)
    df_a=pd.concat([df_a,df])

In [249]:
df_a = df_a[df_a['Value'].notna()]

In [250]:
len(df_a['Product Group'].unique())


51

In [ ]:
df_a

In [254]:
chunk_size=1000000
# Create a list of DataFrames by splitting the original DataFrame
df_chunks = [df_a.iloc[i:i + chunk_size] for i in range(0, len(df_a), chunk_size)]


with pd.ExcelWriter(OFolder+'\JNP_Extract_Consolidated1.xlsx', engine='xlsxwriter') as writer:  # doctest: +SKIP
    for i, chunk in enumerate(df_chunks):
        sheet_name = f"Chunk_{i+1}"  # Naming each sheet dynamically
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)